This is a seq2seq model, which uses Keras's Sequential Model.

The model contains two Bidirectional LSTM layers, an Embedding layer, and dropout.

This notebook is work in progress.

In [1]:
import os, collections

import pandas as pd
import numpy as np

from sklearn.utils import shuffle

from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding, Bidirectional, RepeatVector, Dropout, TimeDistributed
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

from tensorflow.keras.losses import sparse_categorical_crossentropy

In [2]:
from google.colab import drive

drive.mount('/content/gdrive')
root_path = 'gdrive/My Drive/your_project_folder/' 

Mounted at /content/gdrive


In [3]:
def read_data_from_file(filename, data_dict):

    with open(filename) as fp:
        line = fp.readline()
        while line:
            bo, ch, ve, text = tuple(line.strip().split('\t'))
            words = text.split()
            for w in words:  
                # in the output data, composite placenames have a '_', which cannot be found in the input data
                words_split = w.split('_')               
                for word_split in words_split:
                    data_dict[bo].append(word_split)
        
            line = fp.readline()
            
    return data_dict

In [4]:
input_file = '/content/gdrive/My Drive/data/t-in_voc'
#input_file = 't-in_voc'
input_data = collections.defaultdict(list)

output_file = '/content/gdrive/My Drive/data/t-out'
#output_file = 't-out'
output_data = collections.defaultdict(list)

input_data = read_data_from_file(input_file, input_data)
output_data = read_data_from_file(output_file, output_data)

In [5]:
print(len(input_data['Gen']))
print(len(output_data['Gen']))

20611
20611


In [6]:
input_data['Gen'][0:10]

['B.:R;>CIJT',
 'B.@R@>',
 '>:ELOHIJM',
 '>;T',
 'HAC.@MAJIM',
 'W:>;T',
 'H@>@REY',
 'W:H@>@REY',
 'H@J:T@H',
 'TOHW.']

In [7]:
output_data['Gen'][0:10]

['B-R>CJT/',
 'BR>[',
 '>LH(J(M/JM',
 '>T',
 'H-CMJ(M/(JM',
 'W->T',
 'H->RY/:a',
 'W-H->RY/:a',
 'HJ(H[&TH',
 'THW/']

In [8]:
def make_in_out_sequences(data_dict, sequence_length):
    
    all_sequences = []
    for words_list in data_dict.values():

        for w in range(len(words_list) - sequence_length + 1):
    
            seq = ' '.join([words_list[ind] for ind in list(range(w, w + sequence_length))])
        
            # remove some special signs from output data (':', and '='). These only make the sequences longer.
            seq = seq.replace("=", "").replace(":a", "a").replace(":c", "c").replace(":d", "d").replace(":du", "du")
            all_sequences.append(seq)
        
    return all_sequences

In [9]:
sequence_length = 1

all_in_seqs = make_in_out_sequences(input_data, sequence_length)
all_out_seqs = make_in_out_sequences(output_data, sequence_length)


In [10]:
all_in_seqs[0:10]

['B.:R;>CIJT',
 'B.@R@>',
 '>:ELOHIJM',
 '>;T',
 'HAC.@MAJIM',
 'W:>;T',
 'H@>@REY',
 'W:H@>@REY',
 'H@J:T@H',
 'TOHW.']

In [11]:
print(len(all_in_seqs))
print(len(all_out_seqs))

300676
300676


In [12]:
for i in range(206000,206020):
  print(all_in_seqs[i], '---', all_out_seqs[i])

B.:BOW>@M --- B-!!BW>[/+M
J@BOW> --- !J!BW>[
W.B:Y;>T@M --- W-B-!!(JY>[/T+M
J;Y;>W. --- !J!(JY>[W
W.BAXAG.IJM --- W-B-(H-XG/JM
W.BAM.OW<:ADIJM --- W-B-(H-MW<D/JM
T.IH:JEH --- !T!HJH[
HAM.IN:X@H --- H-MNX(H/H
>;JP@H --- >JP(H/H
LAP.@R --- L-(H-PR/a
W:>;JP@H --- W->JP(H/H
L@>AJIL --- L-(H->JL/a
W:LAK.:B@FIJM --- W-L-(H-KBF/JM
MAT.AT --- MTT/
J@DOW --- JD/+W
W:CEMEN --- W-CMN/
HIJN --- HJN/
L@>;JP@H --- L-(H->JP(H/H
W:KIJ --- W-KJ
JA<:AFEH --- !J!<FH[


In [13]:
def prepare_train_data(input_data, output_data):

    input_seqs = []
    output_seqs = []
    input_chars = set()
    output_chars = set()

    # iterate over all the books
    for seq in range(len(input_data)): 
      
        #if len(output_data[seq]) > 40:
        #  continue
          
        if "*" in input_data[seq]: # cases of ketiv/qere are complicated, just skip them!
          continue

        input_list = list(input_data[seq])

        output_list = list(output_data[seq])
        #output_list = ['\t'] + output_list + ['\n']

        input_seqs.append(input_list)
        output_seqs.append(output_list)

        for input_ch in input_list:
            input_chars.add(input_ch)
        
        for output_ch in output_list:
            output_chars.add(output_ch)
                
    
    input_chars = sorted(list(input_chars))
    output_chars = sorted(list(output_chars))
    
    max_len_input = max([len(seq) for seq in input_seqs])
    max_len_output = max([len(seq) for seq in output_seqs])
    
    # shuffle the data. The model will get the data in small batches, it is preferable if the batches are more or less homogeneous
    # of course the inputs and outputs have to be shuffled identically
    input_seqs, output_seqs = shuffle(input_seqs, output_seqs)
    
    return input_seqs, output_seqs, input_chars, output_chars, max_len_input, max_len_output

In [14]:
def create_dicts(input_voc, output_voc):
    
    # these dicts map the input sequences
    input_idx2char = {}
    input_char2idx = {}
    
    input_idx2char[0] = 'PAD'
    input_char2idx['PAD'] = 0

    for k, v in enumerate(input_voc):
        input_idx2char[k + 1] = v
        input_char2idx[v] = k + 1
     
    # and these dicts map the output sequences of parts of speech
    output_idx2char = {}
    output_char2idx = {}
    
    output_idx2char[0] = 'PAD'
    output_char2idx['PAD'] = 0
    
    for k, v in enumerate(output_voc):
        output_idx2char[k + 1] = v
        output_char2idx[v] = k + 1
        
    return input_idx2char, input_char2idx, output_idx2char, output_char2idx

In [15]:
def pad(x, length=None):

    padded_sequences = pad_sequences(x, length, padding='post')
    
    return padded_sequences

In [55]:
def model_bidirectional(input_shape, output_sequence_length, input_vocab_size, output_vocab_size):

    model = Sequential()
    model.add(Embedding(input_vocab_size + 1, 300, input_shape=input_shape[1:]))
    
    model.add(Bidirectional(LSTM(512, return_sequences=False)))
    model.add(RepeatVector(output_sequence_length))
    model.add(Bidirectional(LSTM(512, return_sequences=True)))
    
    model.add(Dropout(0.2))
    model.add(TimeDistributed(Dense(output_vocab_size + 1, activation='softmax')))

    learning_rate = 1e-3
    model.compile(loss=sparse_categorical_crossentropy,
                  optimizer=Adam(learning_rate),
                  metrics=['accuracy'])
    return model

In [56]:
input_seqs, output_seqs, input_chars, output_chars, max_len_input, max_len_output = prepare_train_data(all_in_seqs, all_out_seqs)
print(len(input_seqs))

299488


In [57]:
input_seqs[0]

['L', 'A', 'J', 'H', 'W', '@', 'H']

In [58]:
input_idx2char, input_char2idx, output_idx2char, output_char2idx = create_dicts(input_chars, output_chars)

In [59]:
input_seqs_num = []
output_seqs_num = []

for seq in input_seqs:
    input_seqs_num.append([input_char2idx[char] for char in seq])
    
for seq in output_seqs:
    output_seqs_num.append([output_char2idx[char] for char in seq])

In [60]:
input_seqs_num = pad(input_seqs_num)
output_seqs_num = pad(output_seqs_num)

# sparse_categorical_crossentropy function requires the labels to be in 3 dims
output_seqs_num = output_seqs_num.reshape(*output_seqs_num.shape, 1)

In [61]:
train_size = 150000

input_seqs_num_train = input_seqs_num[0:train_size]
output_seqs_num_train = output_seqs_num[0:train_size]

In [62]:
max_len_input = max([len(seq) for seq in input_seqs_num_train])
max_len_output = max([len(seq) for seq in output_seqs_num_train])
max_len_output

26

In [63]:
tmp_x = pad(input_seqs_num_train, max_len_output)

rnn_model = model_bidirectional(
    tmp_x.shape,
    max_len_output,
    len(input_chars),
    len(output_chars))
    
rnn_model.summary()
rnn_model.fit(tmp_x, output_seqs_num_train, batch_size=256, epochs=20, validation_split=0.2)


Model: "sequential_2"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
embedding_2 (Embedding)      (None, 26, 300)           9900      
_________________________________________________________________
bidirectional_4 (Bidirection (None, 1024)              3330048   
_________________________________________________________________
repeat_vector_2 (RepeatVecto (None, 26, 1024)          0         
_________________________________________________________________
bidirectional_5 (Bidirection (None, 26, 1024)          6295552   
_________________________________________________________________
dropout_2 (Dropout)          (None, 26, 1024)          0         
_________________________________________________________________
time_distributed_2 (TimeDist (None, 26, 41)            42025     
Total params: 9,677,525
Trainable params: 9,677,525
Non-trainable params: 0
____________________________________________

In [64]:
print(len(rnn_model.predict(tmp_x[:1])[0]))

26


In [65]:
num_of_predictions = 10

preds = rnn_model.predict(tmp_x[:num_of_predictions])

for i in range(num_of_predictions):
  
    print([output_idx2char[np.argmax(pred)] for pred in preds[i]])
    print(input_seqs[i])
    print(output_seqs[i])
    print(' ')

['L', '-', 'J', 'H', 'W', 'H', '/', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD']
['L', 'A', 'J', 'H', 'W', '@', 'H']
['L', '-', 'J', 'H', 'W', 'H', '/']
 
['W', '-', 'M', 'F', '>', '/', '>', '[', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD']
['W', '.', 'M', 'A', 'F', '.', 'O', '>']
['W', '-', 'M', 'F', '>', '/']
 
['W', '-', '<', 'L', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD']
['W', ':', '<', 'A', 'L']
['W', '-', '<', 'L']
 
[']', 'N', ']', 'F', 'G', 'B', '[', '/', 'a', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD']
['N', 'I', 'F', ':', 'G', '.', '@', 'B']
[']', 'N', ']', 'F', 'G', 'B', '[', '/', 'a']
 
['>', 'T', 'PAD', 'PAD', 'PAD', 'PAD',